In [ ]:
import os
import shutil
import numpy as np
import torch
import torch.nn.functional as F
from tqdm import tqdm

# ================= 配置 =================
DATASET_ROOT = "./dataset_RAW_IQ"
SIM_TRAIN_DIR = os.path.join(DATASET_ROOT, "sim_train")
SIM_VAL_DIR = os.path.join(DATASET_ROOT, "sim_val")
REAL_TEST_DIR = os.path.join(DATASET_ROOT, "real_test")

CLASS_NAMES = [
    "BPSK", "QPSK", "8PSK", "OQPSK", "PI4DQPSK", 
    "16QAM", "32QAM", "64QAM", 
    "2ASK", "16APSK", "32APSK", "FSK2"
]

# 真实数据路径
FILE_MAPPING = {
    "E:/Project/AMR-Benchmark/PSK8-03-24-25-10h54m48s14.iq": "8PSK",
    "E:/Project/AMR-Benchmark/QAM16-03-24-25-10h52m09s762.iq": "16QAM",
    "E:/Project/AMR-Benchmark/QAM64-03-24-25-10h51m51s805.iq": "64QAM",
    "E:/Project/AMR-Benchmark/ASK2-03-24-25-10h56m09s501.iq": "2ASK",
    "E:/Project/AMR-Benchmark/BPSK-03-24-25-10h52m32s937.iq": "BPSK",
    "E:/Project/AMR-Benchmark/L20250425_103838_F6000000000_S400000_B100000_M0_IQ_ID0_nozaoshen_100ksps_QPSK.dat": "QPSK",
    "E:/Project/AMR-Benchmark/FSK2DEV625KHZ-03-24-25-11h08m50s462.iq": "FSK2",
    "E:/Project/AMR-Benchmark/L20250425_104306_F6000000000_S400000_B100000_M0_IQ_ID0_nojiazao_100ksps_OQPSK.dat": "OQPSK",
    "E:/Project/AMR-Benchmark/PI4DQPSK-03-24-25-10h54m23s982.iq": "PI4DQPSK"
}

# [修改] 输入长度改为 1024 (1D网络不需要太长，太长反而难训练)
# 我们把 10240 的数据切成 10 个 1024
SLICE_LEN = 1024 
TOTAL_SIM_SAMPLES_PER_CLASS = 1000 # 样本数可以多一点
TRAIN_VAL_RATIO = 0.8

# ... (generate_rrc_filter, apply_channel_effects, 调制函数等保持不变，直接复制之前的) ...
# 为节省篇幅，此处省略调制函数定义，请把之前代码里的 copy 过来
# 确保 generate_sim_signal 函数存在

# -----------------------------------------------------------
# [必须把之前的辅助函数和 generate_sim_signal 复制到这里]
# -----------------------------------------------------------
# (此处省略 generate_sim_signal 的具体实现，请复用之前的代码)
# 唯一需要修改的是 generate_sim_signal 中的 N 参数，传入 SLICE_LEN
# 并且保留 step 随机降采样的逻辑
def generate_rrc_filter(N_taps, beta, sps):
    t = torch.arange(-(N_taps - 1) // 2, (N_taps - 1) // 2 + 1, dtype=torch.float32) / sps
    mid_idx = (N_taps - 1) // 2
    t[mid_idx] = 1e-8
    num = torch.sin(np.pi * t * (1 - beta)) + 4 * beta * t * torch.cos(np.pi * t * (1 + beta))
    den = np.pi * t * (1 - (4 * beta * t)**2)
    coeffs = num / den
    coeffs[mid_idx] = 1 - beta + (4 * beta / np.pi)
    return coeffs / torch.sqrt(torch.sum(coeffs**2))

def apply_channel_effects(sig, snr_db):
    N = len(sig)
    freq_offset = np.random.uniform(-50, 50) 
    t = torch.arange(N, dtype=torch.float32) / 80e3
    phase_offset = np.random.uniform(0, 2*np.pi)
    rotation = torch.exp(1j * (2 * np.pi * freq_offset * t + phase_offset))
    sig = sig * rotation
    phase_jitter = torch.randn(N) * 0.005 
    sig = sig * torch.exp(1j * phase_jitter)
    g_imbalance = np.random.uniform(0.98, 1.02)
    p_imbalance = np.random.uniform(-1, 1) * np.pi/180
    real = sig.real * g_imbalance
    imag = sig.imag * np.cos(p_imbalance) - sig.real * np.sin(p_imbalance)
    sig = torch.complex(real, imag)
    sig_power = torch.mean(torch.abs(sig)**2)
    real_snr = snr_db + 5 
    noise_power = sig_power / (10**(real_snr/10.0))
    noise = (torch.randn_like(sig) + 1j*torch.randn_like(sig)) * torch.sqrt(noise_power/2)
    return sig + noise

# 简化的仿真入口
def generate_sim_data_wrapper(cls_name, length, snr):
    # 这里你需要把之前完整的 generate_sim_signal 拷过来
    # 为了演示，假设你已经有了这个函数
    # 记得保留 step = np.random.choice([1, 2, 4]) 的降采样逻辑
    # 这里用伪代码代替，请务必填充完整
    from data_generator_rad_final import generate_sim_signal # 偷个懒，从你之前的文件导入
    
    sig = generate_sim_signal(cls_name, length, snr)
    if isinstance(sig, np.ndarray): sig = torch.from_numpy(sig)
    return sig

def process_iq_tensor(iq_data):
    # 输入: (N,) complex
    # 输出: (2, N) float
    if iq_data.dim() > 1: iq_data = iq_data.squeeze()
    
    # 归一化 (Z-Score normalization per sample is best for 1D-CNN)
    # 或者简单的功率归一化
    pwr = torch.sqrt(torch.mean(iq_data.real**2 + iq_data.imag**2)) + 1e-9
    iq_data = iq_data / pwr
    
    # 堆叠成 (2, N)
    return torch.stack([iq_data.real, iq_data.imag], dim=0).float()

if __name__ == "__main__":
    # 导入之前的生成逻辑
    from data_generator_rad_final import generate_sim_signal
    
    if os.path.exists(DATASET_ROOT): shutil.rmtree(DATASET_ROOT)
    print("===== Generating RAW IQ Data (1D) =====")
    
    # 1. 仿真
    for cls_name in CLASS_NAMES:
        os.makedirs(os.path.join(SIM_TRAIN_DIR, cls_name), exist_ok=True)
        os.makedirs(os.path.join(SIM_VAL_DIR, cls_name), exist_ok=True)
        print(f"Generating {cls_name}...")
        for i in tqdm(range(TOTAL_SIM_SAMPLES_PER_CLASS)):
            target_dir = SIM_TRAIN_DIR if np.random.rand() < TRAIN_VAL_RATIO else SIM_VAL_DIR
            target_dir = os.path.join(target_dir, cls_name)
            snr = np.random.uniform(-5, 25)
            
            # 生成
            sig = generate_sim_signal(cls_name, SLICE_LEN, snr)
            if isinstance(sig, np.ndarray): sig = torch.from_numpy(sig)
            
            # 处理成 (2, 1024)
            tensor = process_iq_tensor(sig)
            torch.save(tensor, os.path.join(target_dir, f"{cls_name}_{i:04d}_{snr:.1f}.pt"))

    # 2. 真实数据
    print("\nProcessing Real Data...")
    for file_path, label in FILE_MAPPING.items():
        if not os.path.exists(file_path): continue
        save_dir = os.path.join(REAL_TEST_DIR, label)
        os.makedirs(save_dir, exist_ok=True)
        try:
            if file_path.endswith('.dat'):
                raw = np.fromfile(file_path, dtype='>i2')
                if len(raw) > 1024: raw = raw[1024:]
            else:
                raw = np.fromfile(file_path, dtype=np.int16)
            
            if raw.size % 2 != 0: raw = raw[:-1]
            I = raw[::2].astype(np.float32)
            Q = raw[1::2].astype(np.float32)
            iq_complex = (I + 1j * Q)
            
            # 切片 (滑动窗口或直接切断)
            # 这里把长数据切成多个 1024 的片段
            num_slices = len(iq_complex) // SLICE_LEN
            for i in range(num_slices):
                if i >= 100: break # 每个文件最多取100个样本
                
                chunk = iq_complex[i*SLICE_LEN : (i+1)*SLICE_LEN]
                chunk_tensor = torch.from_numpy(chunk)
                
                # 处理
                tensor = process_iq_tensor(chunk_tensor)
                torch.save(tensor, os.path.join(save_dir, f"real_{i:04d}.pt"))
                
        except Exception as e: print(f"Error {label}: {e}")